# Sheet 1.2: Tensors and autodiff, PyTorch vs TensorFlow

**Roadmap topic 1.2** (Fanout 03.01-03.09, 04.01, 04.04, 04.05, 04.19). Read first: PyTorch *Learn the Basics*, TF *quickstart for experts*, TF guides on autodiff and `tf.function`, d2l.ai ch. 2-5 (switch between the PyTorch and TensorFlow tabs).

Also clone and solve **Sasha Rush's Tensor Puzzles** (`github.com/srush/Tensor-Puzzles`, open the notebook in Colab). This sheet has its own puzzles, but do those 21 too.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Runs on CPU; the benchmark in Exercise 4 needs a Colab T4.

In [ ]:
import math, time, numpy as np, torch
import torch._dynamo  # load before TensorFlow: avoids an LLVM symbol clash in some torch+TF builds
import tensorflow as tf
torch.manual_seed(0); tf.random.set_seed(0)

## Exercise 1: Puzzles (broadcasting only)

Each function must be **one line** using only `torch.arange`, indexing, comparison, arithmetic, `torch.where`, `.sum()` and broadcasting. No loops, no `torch.cumsum`, `torch.triu`, `torch.eye`, `torch.diag`, `torch.flip`, `torch.bincount`, `torch.repeat_interleave`.

In [ ]:
def eye(n): raise NotImplementedError                 # identity matrix (int)
def triu(n): raise NotImplementedError                # upper-triangular ones incl. diagonal (int)
def cumsum(a): raise NotImplementedError              # 1-D prefix sum
def flip(a): raise NotImplementedError                # reverse a 1-D tensor
def bincount(a, n): raise NotImplementedError         # counts of 0..n-1 in a 1-D int tensor
def repeat(a, k): raise NotImplementedError           # [a0]*k[0] + [a1]*k[1] + ...  (hint: use cumsum and bincount ideas)
def sliding_sum(a, w): raise NotImplementedError      # sums over windows of size w, len(a)-w+1 outputs
def onehot(a, n): raise NotImplementedError           # (len(a), n) int one-hot

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    a = torch.randint(-5, 6, (13,)); ai = torch.randint(0, 7, (40,)); k = torch.randint(0, 4, (9,))
    assert torch.equal(eye(5), torch.eye(5, dtype=torch.long)), "eye"
    assert torch.equal(triu(6), torch.triu(torch.ones(6, 6, dtype=torch.long))), "triu"
    assert torch.equal(cumsum(a), a.cumsum(0)), "cumsum"
    assert torch.equal(flip(a), a.flip(0)), "flip"
    assert torch.equal(bincount(ai, 7), torch.bincount(ai, minlength=7)), "bincount"
    assert torch.equal(repeat(torch.arange(9), k), torch.repeat_interleave(torch.arange(9), k)), "repeat"
    assert torch.equal(sliding_sum(a, 4), a.unfold(0, 4, 1).sum(1)), "sliding_sum"
    assert torch.equal(onehot(ai, 7), torch.nn.functional.one_hot(ai, 7)), "onehot"
    print("PASS  8/8 puzzles")
_check1()

## Exercise 2: Strides, views and copies

For each case, predict whether `y` **shares memory** with `x` (`True`/`False`), or write `"error"` if the line raises. Then run it to learn why. Fill `predictions` before running the check.

In [ ]:
cases = {
    "a": "x = torch.arange(12).view(3, 4); y = x.t()",
    "b": "x = torch.arange(12).view(3, 4); y = x.t().view(-1)",
    "c": "x = torch.arange(12).view(3, 4); y = x.t().reshape(-1)",
    "d": "x = torch.arange(12).view(3, 4); y = x[:, 1]",
    "e": "x = torch.arange(12).view(3, 4); y = x[:, [1, 2]]",
    "f": "x = torch.arange(12).view(3, 4); y = x.t().contiguous()",
    "g": "x = torch.arange(12).view(3, 4); y = x.unsqueeze(0).expand(5, 3, 4)",
    "h": "x = torch.arange(12).view(3, 4); y = x[::2].flatten()",
}

In [ ]:
predictions = {k: None for k in cases}   # YOUR ANSWERS: True, False or "error"

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    wrong = []
    for k, src in cases.items():
        env = {"torch": torch}
        try:
            exec(src, env); x, y = env["x"], env["y"]
            truth = y.untyped_storage().data_ptr() == x.untyped_storage().data_ptr()
        except RuntimeError: truth = "error"
        if predictions[k] != truth: wrong.append(k)
    assert not wrong, f"wrong: {wrong}. Print x.stride(), y.stride() and y.is_contiguous() for those."
    print("PASS  8/8")
_check2()

**Written (2a):** Why does `x.t().view(-1)` fail while `x.t().reshape(-1)` works? What does `.contiguous()` cost, and where in a transformer does this bite you (hint: splitting heads after a transpose)?

**Your answer:**

## Exercise 3: One CNN, two frameworks, identical numbers

Below are the same small CNN in PyTorch and Keras. Write `copy_torch_to_keras(tm, km)` that copies every weight from the PyTorch model into the Keras model so that **outputs and first-step gradients match to 1e-5**.

Things you will have to get right: conv kernel layout (`OIHW` vs `HWIO`), dense layout (`out×in` vs `in×out`), and the flatten order (`NCHW` vs `NHWC`). Then write `train_step_torch` and `train_step_tf` (SGD, lr 0.1, cross-entropy on logits) using a manual loop in PyTorch and `tf.GradientTape` + `@tf.function` in TF.

In [ ]:
class TorchCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = torch.nn.Conv2d(1, 8, 3, padding=1); self.c2 = torch.nn.Conv2d(8, 16, 3, padding=1)
        self.fc = torch.nn.Linear(16 * 7 * 7, 10)
    def forward(self, x):                                   # x: (N, 1, 28, 28)
        x = torch.nn.functional.max_pool2d(torch.relu(self.c1(x)), 2)
        x = torch.nn.functional.max_pool2d(torch.relu(self.c2(x)), 2)
        return self.fc(x.flatten(1))

def keras_cnn():
    L = tf.keras.layers
    return tf.keras.Sequential([tf.keras.Input((28, 28, 1)),
        L.Conv2D(8, 3, padding="same", activation="relu"), L.MaxPool2D(),
        L.Conv2D(16, 3, padding="same", activation="relu"), L.MaxPool2D(),
        L.Flatten(), L.Dense(10)])

In [ ]:
def copy_torch_to_keras(tm, km):
    # YOUR CODE
    raise NotImplementedError

def train_step_torch(model, x, y, lr=0.1):
    # returns loss (float); updates model in place
    # YOUR CODE
    raise NotImplementedError

def make_train_step_tf(model, lr=0.1):
    # returns a @tf.function step(x, y) -> loss
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    torch.manual_seed(0)
    tm, km = TorchCNN(), keras_cnn(); copy_torch_to_keras(tm, km)
    x = torch.randn(32, 1, 28, 28); y = torch.randint(0, 10, (32,))
    xt = tf.constant(x.permute(0, 2, 3, 1).numpy()); yt = tf.constant(y.numpy())
    out_t, out_k = tm(x).detach().numpy(), km(xt).numpy()
    assert np.allclose(out_t, out_k, atol=1e-5), f"forward mismatch, max diff {np.abs(out_t-out_k).max():.2e}"
    step = make_train_step_tf(km)
    lt = [train_step_torch(tm, x, y) for _ in range(5)]
    lk = [float(step(xt, yt)) for _ in range(5)]
    assert np.allclose(lt, lk, atol=1e-5), f"losses diverge: {lt} vs {lk}"
    assert lt[-1] < lt[0], "loss should go down"
    print("PASS  losses:", [round(v, 5) for v in lt])
_check3()

### On Colab: the full comparison
1. Train both versions on Fashion-MNIST for 5 epochs (PyTorch `DataLoader` vs `tf.data`), plus a third version with Keras `model.fit`. Same init, same data order if you can manage it. Report final test accuracy (should agree within 0.3%).
2. Benchmark one training step on a T4: PyTorch eager, `torch.compile`, TF eager, `tf.function`, `tf.function(jit_compile=True)`. Report median step time over 100 steps and the one-off compile time separately.

In [ ]:
# Colab: training + benchmark. Use torch.cuda.synchronize() before reading timers.

## Exercise 4: Framework decision memo

Write one page (in this cell) as if to a lab lead choosing a framework for **LLM research**. Cover: debugging experience, what you measured above, ecosystem (Hugging Face `transformers`/`trl`/`peft`, vLLM, FlashAttention, JAX as the third option), deployment, and hiring signal. End with a clear choice. You will use your choice for Steps 2-4.

**Your memo:**

## Checkpoint
Take any 50-line PyTorch training snippet (e.g. from the PyTorch tutorials) and port it to TensorFlow in under 30 minutes with matching loss curves. Then do the reverse.

---
### Hints for self-grading
- 2a: `view` needs a stride pattern that can express the new shape over the same memory; a transpose breaks that, so a copy is required and only `reshape` will make one. `.contiguous()` is a full memory copy. In attention, `x.view(B,T,H,d).transpose(1,2)` is fine, but merging heads back needs `.transpose(1,2).contiguous().view(B,T,C)` or `.reshape`.